<a href="https://colab.research.google.com/github/zaidlameer/DeetectorPrototype/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install onnxruntime scikit-learn numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


In [7]:
import os
from glob import glob

# Define the base directory containing 'fake' and 'real' folders
base_dir = '/content/drive/MyDrive/dataset-deepfake-images/Dataset/Test'

# Get image paths
fake_images = glob(os.path.join(base_dir, 'Fake', '*'))
real_images = glob(os.path.join(base_dir, 'Real', '*'))

# Combine image paths
image_paths = fake_images + real_images

# Create labels (0 for real, 1 for fake)
labels = [1] * len(fake_images) + [0] * len(real_images)

# Shuffle the dataset
from sklearn.utils import shuffle
image_paths, labels = shuffle(image_paths, labels, random_state=42)

print(f"Total images: {len(image_paths)}")
print(f"Fake images: {len(fake_images)}")
print(f"Real images: {len(real_images)}")


Total images: 10905
Fake images: 5492
Real images: 5413


In [3]:
from PIL import Image
import numpy as np

def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((224, 224))  # Adjust size to match model input
    image_array = np.array(image).astype(np.float32) / 255.0
    image_array = np.transpose(image_array, (2, 0, 1))  # HWC to CHW
    return np.expand_dims(image_array, axis=0)


In [5]:
import onnxruntime as ort

ort_session = ort.InferenceSession("/content/drive/MyDrive/Models/final_deepfake_detector.onnx")
input_name = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name


In [21]:
import os
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def evaluate_model(image_paths, labels):
    predictions = []
    for image_path in image_paths:
        input_data = preprocess_image(image_path)
        ort_inputs = {input_name: input_data}
        ort_outputs = ort_session.run([output_name], ort_inputs)
        predicted_class = np.argmax(ort_outputs[0])
        predictions.append(predicted_class)

    accuracy = accuracy_score(labels, predictions)
    #precision = precision_score(labels, predictions, average='weighted')
    #recall = recall_score(labels, predictions, average='weighted')
    f1 = f1_score(labels, predictions, average='weighted')

    return accuracy, precision, recall, f1

# Assuming you have lists of image_paths and corresponding labels
accuracy, precision, recall, f1 = evaluate_model(image_paths, labels)
print(f"Accuracy: {accuracy:.4f}")
#print(f"Precision: {precision:.4f}")
#print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


KeyError: '__reduce_cython__'